In [2]:
import json
from pathlib import Path

data_path = Path("../data/products.json")

with data_path.open("r", encoding="utf-8") as file:
    products = json.load(file)

print(f"Loaded {len(products)} products")
print(products[0])


Loaded 12 products
{'id': 'cl-001', 'name': 'Classic White Cotton T-Shirt', 'category': 't-shirt', 'gender': 'unisex', 'colors': ['white'], 'style': ['casual', 'basic'], 'material': ['cotton'], 'season': ['spring', 'summer'], 'price': 799, 'rating': 4.6, 'popularity': 98, 'sizes': ['S', 'M', 'L', 'XL'], 'stock': 42, 'imageUrl': 'https://images.unsplash.com/photo-1521572163474-6864f9cf17ab?auto=format&fit=crop&w=600&q=80'}


In [3]:
import pandas as pd

products_df = pd.DataFrame(products)

print("Rows and columns:", products_df.shape)
print("Available fields:")
print(products_df.columns.tolist())

products_df.head(3)


Rows and columns: (12, 14)
Available fields:
['id', 'name', 'category', 'gender', 'colors', 'style', 'material', 'season', 'price', 'rating', 'popularity', 'sizes', 'stock', 'imageUrl']


,id,name,category,gender,colors,style,material,season,price,rating,popularity,sizes,stock,imageUrl
0,cl-001,Classic White Cotton T-Shirt,t-shirt,unisex,[white],"[casual, basic]",[cotton],"[spring, summer]",799,4.6,98,"[S, M, L, XL]",42,https://images.unsplash.com/photo-152157216347...
1,cl-002,Oversized Black Graphic T-Shirt,t-shirt,unisex,[black],"[casual, streetwear, oversized]",[cotton],"[spring, summer, autumn]",999,4.5,94,"[S, M, L, XL]",31,https://images.unsplash.com/photo-150334150425...
2,cl-003,Sky Blue Linen Shirt,shirt,men,[blue],"[casual, smart-casual]",[linen],"[spring, summer]",1599,4.7,91,"[S, M, L, XL]",19,https://images.unsplash.com/photo-160281031838...


In [4]:
products_df.isnull().sum()



id            0
name          0
category      0
gender        0
colors        0
style         0
material      0
season        0
price         0
rating        0
popularity    0
sizes         0
stock         0
imageUrl      0
dtype: int64

In [5]:
feature_columns = [
    "category",
    "gender",
    "colors",
    "style",
    "material",
    "season"
]

def build_feature_text(row):
    parts = []

    for column in feature_columns:
        value = row[column]

        if isinstance(value, list):
            parts.extend(str(item).lower() for item in value)
        else:
            parts.append(str(value).lower())

    return " ".join(parts)

products_df["feature_text"] = products_df.apply(build_feature_text, axis=1)

products_df[["name", "feature_text"]].head()

,name,feature_text
0,Classic White Cotton T-Shirt,t-shirt unisex white casual basic cotton sprin...
1,Oversized Black Graphic T-Shirt,t-shirt unisex black casual streetwear oversiz...
2,Sky Blue Linen Shirt,shirt men blue casual smart-casual linen sprin...
3,Slim Fit Dark Blue Jeans,jeans unisex blue casual everyday denim cotton...
4,High-Waist Straight Blue Jeans,jeans women blue casual everyday high-waist de...


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

feature_matrix = vectorizer.fit_transform(
    products_df["feature_text"]
)

print("Matrix shape:", feature_matrix.shape)

Matrix shape: (12, 51)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(feature_matrix)
print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (12, 12)


In [8]:
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=products_df["name"],
    columns=products_df["name"]
)

similarity_df.round(2)

name,Classic White Cotton T-Shirt,Oversized Black Graphic T-Shirt,Sky Blue Linen Shirt,Slim Fit Dark Blue Jeans,High-Waist Straight Blue Jeans,Floral Summer Midi Dress,Elegant Black Evening Dress,Warm Beige Knit Sweater,Olive Lightweight Bomber Jacket,Navy Blue Zip Hoodie,Relaxed Cotton Cargo Pants,Soft Lavender Lounge Set
name,,,,,,,,,,,,
Classic White Cotton T-Shirt,1.00,0.48,0.34,0.33,0.20,0.30,0.00,0.10,0.03,0.35,0.32,0.09
Oversized Black Graphic T-Shirt,0.48,1.00,0.30,0.33,0.21,0.13,0.18,0.12,0.18,0.19,0.45,0.11
Sky Blue Linen Shirt,0.34,0.30,1.00,0.29,0.24,0.14,0.00,0.05,0.19,0.05,0.17,0.05
Slim Fit Dark Blue Jeans,0.33,0.33,0.29,1.00,0.77,0.12,0.09,0.17,0.12,0.24,0.31,0.16
High-Waist Straight Blue Jeans,0.20,0.21,0.24,0.77,1.00,0.17,0.14,0.09,0.10,0.15,0.20,0.20
Floral Summer Midi Dress,0.30,0.13,0.14,0.12,0.17,1.00,0.18,0.02,0.02,0.02,0.12,0.09
Elegant Black Evening Dress,0.00,0.18,0.00,0.09,0.14,0.18,1.00,0.07,0.08,0.08,0.03,0.14
Warm Beige Knit Sweater,0.10,0.12,0.05,0.17,0.09,0.02,0.07,1.00,0.10,0.15,0.12,0.20
Olive Lightweight Bomber Jacket,0.03,0.18,0.19,0.12,0.10,0.02,0.08,0.10,1.00,0.11,0.32,0.10


In [9]:
def recommend_products(product_id, n=5):
    try:
        product_position = products_df["id"].tolist().index(product_id)
    except ValueError:
        raise ValueError(f"Product ID not found: {product_id}")

    scores = similarity_matrix[product_position]

    ranked_positions = scores.argsort()[::-1]

    ranked_positions = [
        position
        for position in ranked_positions
        if position != product_position
    ][:n]

    recommendations = products_df.iloc[ranked_positions][
        ["id", "name", "category", "colors", "style", "price"]
    ].copy()

    recommendations["similarity_score"] = (
        scores[ranked_positions].round(3)
    )

    return recommendations.reset_index(drop=True)

In [10]:
recommend_products("cl-002", n=5)

,id,name,category,colors,style,price,similarity_score
0,cl-001,Classic White Cotton T-Shirt,t-shirt,[white],"[casual, basic]",799,0.484
1,cl-011,Relaxed Cotton Cargo Pants,trousers,[khaki],"[casual, utility, streetwear]",1999,0.446
2,cl-004,Slim Fit Dark Blue Jeans,jeans,[blue],"[casual, everyday]",2199,0.329
3,cl-003,Sky Blue Linen Shirt,shirt,[blue],"[casual, smart-casual]",1599,0.299
4,cl-005,High-Waist Straight Blue Jeans,jeans,[blue],"[casual, everyday, high-waist]",2399,0.212
